In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
#Flink
import pandas as pd

FILE_PATH="/content/drive/MyDrive/Colab Notebooks/"

# ===============================
# LOAD MAIN STREAM
# ===============================

df=pd.read_csv(FILE_PATH+"windows_main_stream_test.csv")

print("Events loaded:",len(df))

df["date_and_time"]=pd.to_datetime(df["date_and_time"])

df=df.sort_values("date_and_time")

events=df["event_id"].tolist()
times=df["date_and_time"].tolist()

detected=[]


# ===============================
# MITRE ATTACK MAP
# ===============================

MITRE_MAP={

4625:("Brute Force","T1110"),
5379:("Credential Dump","T1003"),
4648:("Token Impersonation","T1134"),
4616:("Time Manipulation","T1070"),
4798:("Account Discovery","T1087"),
4799:("Account Discovery","T1087"),
4672:("Privilege Escalation","T1068")

}


# ===============================
# COOLDOWN CONTROL
# ===============================

GLOBAL_COOLDOWN=600

last_attack_time={}


def register_attack(name,start,end,event):

    if name not in last_attack_time or \
    (start-last_attack_time[name]).total_seconds()>GLOBAL_COOLDOWN:

        technique,tid=MITRE_MAP.get(event,("Unknown","Unknown"))

        detected.append({

        "attack":name,
        "technique":technique,
        "mitre_id":tid,
        "severity":2,
        "start_time":start,
        "end_time":end

        })

        last_attack_time[name]=start


# ===============================
# DETECT ATTACK PATTERNS
# ===============================

for i in range(len(events)-2):

    # Credential dump
    if events[i]==4624 and events[i+1]==4672 and events[i+2]==5379:

        register_attack(
        "Credential Dump",
        times[i],
        times[i+2],
        5379
        )

    # Brute force attack
    if events[i]==4625:

        count=1
        j=i+1

        while j<len(events) and (times[j]-times[i]).total_seconds()<120:

            if events[j]==4625:
                count+=1

            j+=1

        if count>=8:

            register_attack(
            "Brute Force Attack",
            times[i],
            times[j-1],
            4625
            )

    # Account enumeration
    if events[i] in [4798,4799]:

        count=1
        j=i+1

        while j<len(events) and (times[j]-times[i]).total_seconds()<180:

            if events[j] in [4798,4799]:
                count+=1

            j+=1

        if count>=8:

            register_attack(
            "Account Enumeration",
            times[i],
            times[j-1],
            4798
            )

    # Privilege escalation
    if events[i]==4672:

        count=1
        j=i+1

        while j<len(events) and (times[j]-times[i]).total_seconds()<120:

            if events[j]==4672:
                count+=1

            j+=1

        if count>=5:

            register_attack(
            "Privilege Escalation",
            times[i],
            times[j-1],
            4672
            )

    # Time manipulation
    if events[i]==4616:

        register_attack(
        "Time Manipulation",
        times[i],
        times[i],
        4616
        )


# ===============================
# SAVE DETECTIONS
# ===============================

attack_df=pd.DataFrame(detected)

print("\nAttacks detected:",len(attack_df))

#attack_df.to_csv(FILE_PATH+"windows_attack_patterns.csv",index=False)
attack_df.to_csv(
FILE_PATH+"windows_attack_patterns.csv",
index=False
)

print("Threat intelligence saved")


# ===============================
# SOC INCIDENT REPORT
# ===============================

print("\n========== SOC INCIDENT REPORT ==========\n")

print("Total incidents:",len(attack_df))

print("\nAttack distribution")

print(attack_df["attack"].value_counts())


# Most active hour

attack_df["hour"]=pd.to_datetime(
attack_df["start_time"]
).dt.hour

print("\nMost active attack hours")

print(attack_df["hour"].value_counts().head())

Events loaded: 3022

Attacks detected: 64
Threat intelligence saved

========== SOC INCIDENT REPORT ==========

Total incidents: 64

Attack distribution
attack
Time Manipulation    61
Credential Dump       3
Name: count, dtype: int64

Most active attack hours
hour
1     7
17    6
18    6
14    5
19    5
Name: count, dtype: int64


In [ ]:
# import pandas as pd

# FILE_PATH="/content/drive/MyDrive/Mini_Project/"

# df=pd.read_csv(FILE_PATH+"windows_ai_output.csv")

# print("Events loaded:",len(df))

# # ===============================
# # STREAM SPLIT
# # ===============================

# main_stream=df[df["risk_score"]==0]

# dlq_stream=df[df["risk_score"]==1]

# print("Main stream logs:",len(main_stream))
# print("DLQ logs:",len(dlq_stream))

# # ===============================
# # SAVE STREAMS
# # ===============================

# main_stream.to_csv(

# FILE_PATH+"windows_main_stream.csv",
# index=False

# )

# dlq_stream.to_csv(

# FILE_PATH+"windows_dlq_stream.csv",
# index=False

# )

# print("Streams saved")

Events loaded: 84065

Running Threat Intelligence Analysis...

Attacks detected: 171

⚠ ATTACK DETECTED
Attack: Credential Dump Attack
Technique: Credential Dump
MITRE: T1003
Severity: HIGH_RISK
Start: 2024-09-25 23:01:17

⚠ ATTACK DETECTED
Attack: Credential Dump Attack
Technique: Credential Dump
MITRE: T1003
Severity: HIGH_RISK
Start: 2024-10-04 17:06:00

⚠ ATTACK DETECTED
Attack: Credential Dump Attack
Technique: Credential Dump
MITRE: T1003
Severity: HIGH_RISK
Start: 2024-10-05 19:30:28

⚠ ATTACK DETECTED
Attack: Credential Dump Attack
Technique: Credential Dump
MITRE: T1003
Severity: HIGH_RISK
Start: 2024-10-09 18:46:51

⚠ ATTACK DETECTED
Attack: Credential Dump Attack
Technique: Credential Dump
MITRE: T1003
Severity: HIGH_RISK
Start: 2024-10-11 13:40:31

⚠ ATTACK DETECTED
Attack: Credential Dump Attack
Technique: Credential Dump
MITRE: T1003
Severity: HIGH_RISK
Start: 2024-10-14 16:25:12

⚠ ATTACK DETECTED
Attack: Credential Dump Attack
Technique: Credential Dump
MITRE: T1003
Sev

In [ ]:
# import pandas as pd

# FILE_PATH="/content/drive/MyDrive/Colab Notebooks/"

# df=pd.read_csv(FILE_PATH+"windows_gatekeeper_logs.csv",low_memory=False)

# print("Events loaded:",len(df))


# # ===============================
# # TIMESTAMP
# # ===============================

# df["Date_and_Time"]=pd.to_datetime(df["Date_and_Time"],errors="coerce")

# df=df.sort_values("Date_and_Time")

# df["Event_ID"]=df["Event_ID"].astype(int)

# events=df["Event_ID"].tolist()
# times=df["Date_and_Time"].tolist()
# risk=df["risk_score"].tolist()


# # ===============================
# # GLOBAL ALERT COOLDOWN
# # ===============================

# GLOBAL_COOLDOWN=600   # 10 minutes

# last_attack_time={}


# # ===============================
# # MITRE ATTACK MAP
# # ===============================

# MITRE_MAP={

# 4625:("Brute Force","T1110"),
# 5379:("Credential Dump","T1003"),
# 4648:("Token Impersonation","T1134"),
# 4616:("Time Manipulation","T1070"),
# 4798:("Account Discovery","T1087"),
# 4799:("Account Discovery","T1087"),
# 4672:("Privilege Escalation","T1068")

# }


# # ===============================
# # ATTACK RULES
# # ===============================

# attack_rules={

# "Credential Dump Attack":{
# "login_event":4624,
# "priv_event":4672,
# "target_event":5379,
# "min_occurrences":40,
# "time_window":120
# },

# "Brute Force Login":{
# "fail_event":4625,
# "success_event":4624,
# "min_failures":8,
# "time_window":180
# },

# "Account Enumeration":{
# "target_event":4798,
# "min_occurrences":12,
# "time_window":180
# },

# "Token Impersonation":{
# "login_event":4624,
# "token_event":4648,
# "priv_event":4672,
# "time_window":120
# },

# "Time Manipulation":{
# "target_event":4616,
# "min_occurrences":3,
# "time_window":120
# }

# }


# print("\nRunning Threat Intelligence Analysis...\n")

# detected=[]


# # ======================================================
# # HELPER FUNCTION
# # ======================================================

# def register_attack(name,start,end,score,extra=None):

#     if name not in last_attack_time or \
#     (start-last_attack_time[name]).total_seconds()>GLOBAL_COOLDOWN:

#         technique="Unknown"
#         tid="Unknown"

#         if extra and "event" in extra:
#             if extra["event"] in MITRE_MAP:
#                 technique,tid=MITRE_MAP[extra["event"]]

#         severity="LOW"

#         if score>0.85:
#             severity="CRITICAL"
#         elif score>0.70:
#             severity="MEDIUM"

#         detected.append({
#         "attack":name,
#         "technique":technique,
#         "mitre_id":tid,
#         "severity":severity,
#         "start_time":start,
#         "end_time":end
#         })

#         last_attack_time[name]=start


# # ======================================================
# # CREDENTIAL DUMP
# # ======================================================

# rule=attack_rules["Credential Dump Attack"]

# i=0

# while i<len(events):

#     if events[i]!=rule["login_event"]:
#         i+=1
#         continue

#     start=times[i]

#     priv=False
#     count=0
#     j=i+1

#     while j<len(events):

#         if (times[j]-start).total_seconds()>rule["time_window"]:
#             break

#         if events[j]==rule["priv_event"]:
#             priv=True

#         if events[j]==rule["target_event"]:
#             count+=1

#         j+=1

#     avg_risk=sum(risk[i:j])/max(1,(j-i))

#     if priv and count>=rule["min_occurrences"] and avg_risk>0.80:

#         register_attack(
#         "Credential Dump Attack",
#         start,
#         times[j-1],
#         avg_risk,
#         {"event":5379}
#         )

#         i=j
#         continue

#     i+=1


# # ======================================================
# # BRUTE FORCE
# # ======================================================

# rule=attack_rules["Brute Force Login"]

# i=0

# while i<len(events):

#     if events[i]!=rule["fail_event"]:
#         i+=1
#         continue

#     start=times[i]

#     failures=1
#     success=False
#     j=i+1

#     while j<len(events):

#         if (times[j]-start).total_seconds()>rule["time_window"]:
#             break

#         if events[j]==rule["fail_event"]:
#             failures+=1

#         if events[j]==rule["success_event"]:
#             success=True

#         j+=1

#     avg_risk=sum(risk[i:j])/max(1,(j-i))

#     if failures>=rule["min_failures"] and success and avg_risk>0.70:

#         register_attack(
#         "Brute Force Login",
#         start,
#         times[j-1],
#         avg_risk,
#         {"event":4625}
#         )

#         i=j
#         continue

#     i+=1


# # ======================================================
# # ACCOUNT ENUMERATION
# # ======================================================

# rule=attack_rules["Account Enumeration"]

# i=0

# while i<len(events):

#     if events[i]!=rule["target_event"]:
#         i+=1
#         continue

#     start=times[i]

#     count=1
#     j=i+1

#     while j<len(events):

#         if (times[j]-start).total_seconds()>rule["time_window"]:
#             break

#         if events[j]==rule["target_event"]:
#             count+=1

#         j+=1

#     avg_risk=sum(risk[i:j])/max(1,(j-i))

#     if count>=rule["min_occurrences"] and avg_risk>0.65:

#         register_attack(
#         "Account Enumeration",
#         start,
#         times[j-1],
#         avg_risk,
#         {"event":4798}
#         )

#         i=j
#         continue

#     i+=1


# # ======================================================
# # TOKEN IMPERSONATION
# # ======================================================

# rule=attack_rules["Token Impersonation"]

# for i in range(len(events)):

#     if events[i]!=rule["login_event"]:
#         continue

#     start=times[i]

#     token=False
#     priv=False

#     j=i+1

#     while j<len(events):

#         if (times[j]-start).total_seconds()>rule["time_window"]:
#             break

#         if events[j]==rule["token_event"]:
#             token=True

#         if events[j]==rule["priv_event"]:
#             priv=True

#         j+=1

#     if token and priv:

#         register_attack(
#         "Token Impersonation",
#         start,
#         times[j-1],
#         risk[i],
#         {"event":4648}
#         )


# # ======================================================
# # TIME MANIPULATION
# # ======================================================

# rule=attack_rules["Time Manipulation"]

# i=0

# while i<len(events):

#     if events[i]!=rule["target_event"]:
#         i+=1
#         continue

#     start=times[i]

#     count=1
#     j=i+1

#     while j<len(events):

#         if (times[j]-start).total_seconds()>rule["time_window"]:
#             break

#         if events[j]==rule["target_event"]:
#             count+=1

#         j+=1

#     if count>=rule["min_occurrences"]:

#         register_attack(
#         "Time Manipulation",
#         start,
#         times[j-1],
#         risk[i],
#         {"event":4616}
#         )

#         i=j
#         continue

#     i+=1


# # ===============================
# # RESULTS
# # ===============================

# print("Attacks detected:",len(detected))

# for a in detected[:10]:

#     print("\n⚠ ATTACK DETECTED")
#     print("Attack:",a["attack"])
#     print("Technique:",a["technique"])
#     print("MITRE:",a["mitre_id"])
#     print("Severity:",a["severity"])
#     print("Start:",a["start_time"])


# # ===============================
# # SAVE
# # ===============================

# attack_df=pd.DataFrame(detected)

# attack_df.to_csv(
# FILE_PATH+"windows_attack_patterns.csv",
# index=False
# )

# print("\nThreat intel saved.")

